# Salary Linear Regression Training Reference

Notebook nay minh hoa baseline `sklearn.linear_model.LinearRegression` cho du bao luong IT tu `data/analysis/salary_analysis_clean.csv`.

Workflow train chinh nen chay tren may local bang CLI hoac `scripts/train_salary_regression.ps1`. Notebook nay chi la reference/review de xem tung buoc.

## Caveats

- Day la baseline de hoc va demo, khong phai ket luan luong thi truong.
- Dataset salary hien chi gom job co numeric salary. TopDev hien khong co numeric salary nen khong vao model.
- Target dung `log_salary_monthly_vnd` de giam skew va outlier influence.
- Coefficient cua Linear Regression chi la tin hieu trong model, khong phai bang chung nhan qua.

## 0. Setup

In [1]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "analysis" / "salary_analysis_clean.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing data/analysis/salary_analysis_clean.csv")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SALARY_PATH = REPO_ROOT / "data" / "analysis" / "salary_analysis_clean.csv"
OUTPUT_DIR = REPO_ROOT / "data" / "modeling" / "salary_regression"

print(f"Repo root: {REPO_ROOT}")
print(f"Salary input: {SALARY_PATH}")
print(f"Model output dir: {OUTPUT_DIR}")

Repo root: G:\project\Vietnam IT Job Market Intelligence
Salary input: G:\project\Vietnam IT Job Market Intelligence\data\analysis\salary_analysis_clean.csv
Model output dir: G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression


In [2]:
from modeling.salary_regression import (
    ARTIFACT_FILENAMES,
    MODEL_FILENAME,
    fit_salary_linear_regression,
    prepare_salary_modeling_data,
    write_outputs,
)

## 1. Load Data

Cell nay kiem tra nhanh dung file, so dong va coverage source/currency truoc khi train.

In [3]:
salary = pd.read_csv(SALARY_PATH, encoding="utf-8-sig")

print(f"Rows: {len(salary):,}")
print(f"Unique URLs: {salary['url'].nunique():,}")
assert len(salary) > 0, "salary_analysis_clean.csv must contain rows"
assert salary["url"].is_unique, "salary dataset should be deduped by URL"

display(salary[["source", "title", "company", "location", "salary_raw", "salary_currency", "salary_period", "salary_midpoint", "seniority", "experience_min"]].head(10))
display(pd.crosstab(salary["source"], salary["salary_currency"], dropna=False))

Rows: 504
Unique URLs: 504


,source,title,company,location,salary_raw,salary_currency,salary_period,salary_midpoint,seniority,experience_min
0,itviec,10 Fullstack Deverloper (Java/ Spring Boot/Angular),Thien Hoang Solutions JSC,Hà Nội,1200 - 1600 USD,USD,month,1400.0,middle,4.0
1,itviec,[10] Nhân sự đánh giá rò quét lỗ hổng ANTT (Pentest),TRUNG TÂM THÔNG TIN TÍN DỤNG QUỐC GIA VIỆT NAM (CIC),Hà Nội,800 - 1500 USD,USD,month,1150.0,middle,2.0
2,itviec,[11] Nhân sự săn tìm mối nguy (Thread huntting),TRUNG TÂM THÔNG TIN TÍN DỤNG QUỐC GIA VIỆT NAM (CIC),Hà Nội,800 - 1500 USD,USD,month,1150.0,middle,2.0
3,itviec,"20 Java Backend Devs (Spring Boot, Hibernate, Oracle)",Thien Hoang Solutions JSC,Hà Nội,1100 - 1900 USD,USD,month,1500.0,middle,3.0
4,itviec,AI Agent Engineer (Contract): Build the Agent Layer,Panto Martech LLC,Hồ Chí Minh,2500 - 4000 USD,USD,month,3250.0,NaN,NaN
5,itviec,"AI Agent Engineer (Python, LLM)",KDS Vietnam,Hồ Chí Minh,1300 - 1800 USD,USD,month,1550.0,middle,3.0
6,itviec,AI Engineer (Định hướng Data Scientist),KALAPA,Hà Nội,600 - 1300 USD,USD,month,950.0,middle,2.0
7,itviec,Middle AI Engineer (Inference Engineering),Autonomous,Hồ Chí Minh,1000 - 2000 USD,USD,month,1500.0,middle,1.0
8,itviec,AI Engineer (Machine Learning/LLM/API),CLUEGA,Hồ Chí Minh,"18M-65,5M",VND,month,41750000.0,middle,2.0
9,itviec,AI Engineer with Leading Korean AI Entrepreneurs,Abc Studio Việt Nam,Hồ Chí Minh,1000 - 2000 USD,USD,month,1500.0,middle,3.0


salary_currency,USD,VND
source,,
itviec,176,26
topcv,6,296


## 2. Prepare Modeling Data

Backend helper lam cac viec chinh: convert salary ve VND/thang, tao target log salary, normalize location/skills, tao feature flags, va chan leakage columns.

In [4]:
modeling_data = prepare_salary_modeling_data(
    salary,
    top_skills=30,
    min_skill_count=5,
    usd_to_vnd=26_000,
)

print(f"Modeling rows: {len(modeling_data.frame):,}")
print(f"Feature count: {len(modeling_data.feature_columns):,}")
print(f"Top skills: {modeling_data.top_skills[:10]}")
display(modeling_data.audit.head(25))
display(modeling_data.frame[["source", "salary_raw", "salary_currency", "salary_period", "salary_period_clean", "salary_monthly_vnd_m", "location_norm", "skill_count", "log_salary_monthly_vnd"]].head(10))

Modeling rows: 504
Feature count: 37
Top skills: ['sql', 'python', 'java', 'docker', 'aws', 'javascript', 'postgresql', 'react', 'devops', 'mysql']


,metric,value
0,input_rows,504.0000
1,modeling_rows,504.0000
2,dropped_invalid_salary_rows,0.0000
3,usd_to_vnd,26000.0000
4,salary_period_year_without_annual_signal,101.0000
5,salary_period_annual_signal_rows,0.0000
6,top_skill_count,30.0000
7,feature_count,37.0000
8,missing_rate_experience_min,0.1389
9,missing_rate_skill_count,0.0000


,source,salary_raw,salary_currency,salary_period,salary_period_clean,salary_monthly_vnd_m,location_norm,skill_count,log_salary_monthly_vnd
0,itviec,1200 - 1600 USD,USD,month,month,36.40,Ha Noi,12,17.410079
1,itviec,800 - 1500 USD,USD,month,month,29.90,Ha Noi,3,17.213369
2,itviec,800 - 1500 USD,USD,month,month,29.90,Ha Noi,3,17.213369
3,itviec,1100 - 1900 USD,USD,month,month,39.00,Ha Noi,14,17.479072
4,itviec,2500 - 4000 USD,USD,month,month,84.50,Ho Chi Minh,7,18.252262
5,itviec,1300 - 1800 USD,USD,month,month,40.30,Ho Chi Minh,7,17.511862
6,itviec,600 - 1300 USD,USD,month,month,24.70,Ha Noi,4,17.022314
7,itviec,1000 - 2000 USD,USD,month,month,39.00,Ho Chi Minh,7,17.479072
8,itviec,"18M-65,5M",VND,month,month,41.75,Ho Chi Minh,10,17.547210
9,itviec,1000 - 2000 USD,USD,month,month,39.00,Ho Chi Minh,6,17.479072


## 3. Train Linear Regression

In [5]:
result = fit_salary_linear_regression(
    modeling_data,
    test_size=0.2,
    random_state=42,
)

print(f"Train rows: {result.train_rows:,}")
print(f"Test rows: {result.test_rows:,}")
display(result.metrics)

Train rows: 403
Test rows: 101


,scope,group_column,group_value,n,mae_log,rmse_log,r2_log,mae_million_vnd,rmse_million_vnd,median_abs_error_million_vnd
0,overall,,,101,0.348026,0.470812,0.513079,9.231780,13.834239,5.458529
1,group,source,itviec,40,0.351169,0.441477,0.472074,13.081841,17.927591,9.790241
2,group,source,topcv,61,0.345965,0.489094,0.339211,6.707150,10.302048,3.921338
3,group,seniority,intern,4,0.520225,0.625492,-0.158547,2.256832,2.576983,2.361662
4,group,seniority,junior,17,0.349652,0.500227,-0.040138,8.290831,13.577077,3.518753
5,group,seniority,lead,12,0.313618,0.398176,0.408171,11.978040,18.724615,5.083154
6,group,seniority,middle,35,0.277802,0.372791,0.312220,7.079958,9.632559,4.558608
7,group,seniority,senior,14,0.370413,0.423634,0.083415,16.900182,21.080537,13.380201
8,group,seniority,Unknown,19,0.444913,0.619427,0.248774,8.121094,11.455665,7.488182


## 4. Inspect Prediction Errors

`abs_error_million_vnd` cho biet job nao model du doan lech nhieu nhat trong test set.

In [6]:
display(result.predictions.head(20))
display(result.predictions["abs_error_million_vnd"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_frame())

,url,title,company,source,location,location_norm,seniority,experience_min,actual_log_salary,predicted_log_salary,actual_salary_million_vnd,predicted_salary_million_vnd,residual_million_vnd,abs_error_million_vnd
0,https://itviec.com/it-jobs/engineering-lead-python-ai-cloud-platform-skylink-labs-5029,"Engineering Lead (Python, AI, Cloud Platform )",Skylink Labs,itviec,Hồ Chí Minh,Ho Chi Minh,lead,NaN,18.252262,17.430920,84.500,37.166558,47.333442,47.333442
1,https://itviec.com/it-jobs/senior-embedded-android-aosp-developer-mijo-connected-vietnam-4927,"Senior ""Embedded Android"" AOSP Developer",MIJO CONNECTED VIETNAM,itviec,Hồ Chí Minh,Ho Chi Minh,senior,6.0,18.326370,17.659172,91.000,46.696134,44.303866,44.303866
2,https://itviec.com/it-jobs/mid-sr-ios-developer-english-required-rakuten-fintech-vietnam-co-ltd-1512,Mid/Sr iOS Developer - English required,"Rakuten Fintech Vietnam Co., Ltd.",itviec,Hồ Chí Minh,Ho Chi Minh,senior,4.0,18.236758,17.523525,83.200,40.772767,42.427233,42.427233
3,https://www.topcv.vn/viec-lam/leader-developer/2220648.html,Leader Developer,Công ty TNHH Quốc Tế Hải Long,topcv,Ha Noi,Ha Noi,junior,1.0,17.822844,16.697283,55.000,17.845875,37.154125,37.154125
4,https://itviec.com/it-jobs/senior-solution-architect-erp-odoo-up-to-80m-vnd-technext-5610,Senior Solution Architect - ERP / Odoo (Up to 80M VND+),TECHNEXT,itviec,Hồ Chí Minh,Ho Chi Minh,lead,2.0,18.197537,17.599768,80.000,44.003005,35.996995,35.996995
5,https://www.topcv.vn/viec-lam/lua-developer-game-thu-nhap-upto-80m-thang-theo-nang-luc-lam-viec-tai-go-vap/2205023.html,"Lua Developer (Game) - Thu Nhập Upto 80M/Tháng Theo Năng Lực, Làm Việc Tại Gò Vấp",CÔNG TY TNHH DỊCH VỤ BÁCH GIA,topcv,Ho Chi Minh,Ho Chi Minh,<NA>,NaN,17.676240,16.507627,47.500,14.762894,32.737106,32.737106
6,https://www.topcv.vn/viec-lam/ky-su-lap-trinh-nhung/1375749.html,Kỹ Sư Lập Trình Nhúng,CÔNG TY TNHH WORLDING VIỆT NAM,topcv,Nước Ngoài,Nước Ngoài,junior,1.0,17.622173,16.465291,45.000,14.150931,30.849069,30.849069
7,https://www.topcv.vn/viec-lam/ky-su-quan-tri-he-thong/2198907.html,Kỹ Sư Quản Trị Hệ Thống,Trung tâm CNTT - BIDV,topcv,Ha Noi,Ha Noi,<NA>,NaN,17.565015,16.674117,42.500,17.437210,25.062790,25.062790
8,https://itviec.com/it-jobs/oq901-fullstack-developer-reactjs-nestjs-nodejs-likelion-vietnam-3531,"OQ901 - Fullstack Developer (ReactJS, NestJS, NodeJS)",LikeLion Vietnam,itviec,Hồ Chí Minh,Ho Chi Minh,senior,5.0,17.296751,17.861212,32.500,57.151234,-24.651234,24.651234
9,https://itviec.com/it-jobs/it-application-implementation-seasonal-1-year-contract-wilmar-clv-cambodia-laos-vietnam-3308,IT Application Implementation (seasonal 1 year contract,Wilmar CLV (Cambodia Laos Vietnam),itviec,Hồ Chí Minh,Ho Chi Minh,middle,3.0,16.618871,17.531231,16.500,41.088176,-24.588176,24.588176


,abs_error_million_vnd
count,101.000000
mean,9.231780
std,10.354806
min,0.046697
25%,2.294133
50%,5.458529
75%,11.968110
90%,23.061655
95%,32.737106
max,47.333442


## 5. Inspect Coefficients

Doc coefficient nhu tin hieu baseline. Feature one-hot va feature correlated khong nen duoc doc nhu quan he nhan qua.

In [7]:
display(result.coefficients.head(30))
display(result.coefficients.sort_values("coefficient", ascending=False).head(15))
display(result.coefficients.sort_values("coefficient", ascending=True).head(15))

,feature,coefficient,abs_coefficient
0,categorical__location_norm_Nhật Bản,1.418749,1.418749
1,categorical__seniority_intern,-1.032532,1.032532
2,categorical__location_norm_Gia Lai,-0.738234,0.738234
3,categorical__location_norm_Bắc Ninh,0.491109,0.491109
4,categorical__location_norm_Tây Ninh,-0.391978,0.391978
5,categorical__location_norm_Cần Thơ,-0.382081,0.382081
6,categorical__seniority_lead,0.378233,0.378233
7,skill__skill__php,0.354506,0.354506
8,categorical__seniority_senior,0.346151,0.346151
9,skill__skill__cloud,-0.336087,0.336087


,feature,coefficient,abs_coefficient
0,categorical__location_norm_Nhật Bản,1.418749,1.418749
3,categorical__location_norm_Bắc Ninh,0.491109,0.491109
6,categorical__seniority_lead,0.378233,0.378233
7,skill__skill__php,0.354506,0.354506
8,categorical__seniority_senior,0.346151,0.346151
11,categorical__location_norm_Thanh Hóa,0.262374,0.262374
14,categorical__seniority_middle,0.216956,0.216956
15,skill__skill__english,0.213993,0.213993
16,skill__skill__aws,0.207075,0.207075
18,categorical__source_itviec,0.176779,0.176779


,feature,coefficient,abs_coefficient
1,categorical__seniority_intern,-1.032532,1.032532
2,categorical__location_norm_Gia Lai,-0.738234,0.738234
4,categorical__location_norm_Tây Ninh,-0.391978,0.391978
5,categorical__location_norm_Cần Thơ,-0.382081,0.382081
9,skill__skill__cloud,-0.336087,0.336087
10,categorical__location_norm_Khánh Hòa,-0.315689,0.315689
12,skill__skill__agile,-0.257541,0.257541
13,categorical__location_norm_Thành phố khác,-0.229574,0.229574
17,skill__skill__tester,-0.202427,0.202427
19,categorical__source_topcv,-0.176779,0.176779


## 6. Save Trained Model For Streamlit

Sau cell nay, Streamlit co the load `model.joblib` trong output folder.

In [8]:
write_outputs(result, OUTPUT_DIR)

artifact_rows = []
for filename in ARTIFACT_FILENAMES:
    path = OUTPUT_DIR / filename
    artifact_rows.append(
        {
            "artifact": filename,
            "exists": path.exists(),
            "path": path,
            "size_kb": round(path.stat().st_size / 1024, 1) if path.exists() else None,
        }
    )

print(f"Notebook trained model file: {OUTPUT_DIR / MODEL_FILENAME}")
display(pd.DataFrame(artifact_rows))

Notebook trained model file: G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression\model.joblib


,artifact,exists,path,size_kb
0,model.joblib,True,G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression\model.joblib,15.7
1,metrics.csv,True,G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression\metrics.csv,1.3
2,predictions_test.csv,True,G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression\predictions_test.csv,32.6
3,coefficients.csv,True,G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression\coefficients.csv,3.9
4,data_audit.csv,True,G:\project\Vietnam IT Job Market Intelligence\data\modeling\salary_regression\data_audit.csv,1.2


## 7. How To Use

Train chinh bang terminal local:

```bash
python -m modeling.salary_regression --input data/analysis/salary_analysis_clean.csv --output-dir data/modeling/salary_regression
# hoac tren Windows
.\scripts\train_salary_regression.ps1
```

Chay UI:

```bash
streamlit run streamlit_salary_regression_opencode.py
```